In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import col
catalog_name='website'

In [0]:
df_bronze_metadata=spark.read.table(f'{catalog_name}.bronze.brz_metadata')
display(df_bronze_metadata.limit(10))


In [0]:
drop_cols = ["user_agent","ip_address","hash","title","search","screenWidth","screenHeight","viewportWidth","viewportHeight"]
silver_df = df_bronze_metadata.drop(*drop_cols)
display(silver_df.limit(10))
display(silver_df.schema)
display(silver_df.describe())
display(silver_df.count())

In [0]:
silver_df=(
            silver_df
           .drop('path','level')
           .withColumn('id',col('id').cast(LongType()))
           .withColumn('user_id',col('user_id').cast(LongType()))
           .withColumn('timestamp',col('timestamp').cast(TimestampType()))
           .withColumn('created_at',col("created_at").cast(TimestampType()))
           .withColumn('duration',col('duration').cast(LongType()))
           )
silver_df.printSchema()

root
 |-- id: long (nullable = true)
 |-- user_id: long (nullable = true)
 |-- event_type: string (nullable = true)
 |-- page_url: string (nullable = true)
 |-- referrer: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- os: string (nullable = true)
 |-- browser: string (nullable = true)
 |-- language: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- device_type: string (nullable = true)
 |-- score: double (nullable = true)
 |-- skill_name: string (nullable = true)
 |-- skill_type: string (nullable = true)
 |-- skill_level: double (nullable = true)
 |-- duration: long (nullable = true)



In [0]:
display(silver_df.groupBy('skill_name').count())
display(silver_df.groupBy('skill_level').count())
display(silver_df.groupBy('referrer').count())

skill_name,count
null,679
Excel for Data Analysis,4
Machine Learning,1
Postgre SQL,2
حمثلاح,2
Ai Automation,2
Power BI,2
PostgreSQL for Data Analysis,2
Python for Data Analysis,2
postgre SQL,4


skill_level,count
null,728


referrer,count
https://vercel.com/,185
https://www.facebook.com/,6
http://m.facebook.com,6
null,158
https://skillsbridge-eight.vercel.app/sw.js,20
,353


In [0]:
silver_df=silver_df.filter(
       col('skill_name').isNull() |
        ~col('skill_name').isin(['حمثلاح','Eating','Sleep','cristiano ronaldo is bad','sleeping','Sleeping']))
display(silver_df.groupBy('skill_name').count())

skill_name,count
null,679
Excel for Data Analysis,4
Machine Learning,1
Postgre SQL,2
Ai Automation,2
Power BI,2
PostgreSQL for Data Analysis,2
Python for Data Analysis,2
postgre SQL,4
Excel,2


In [0]:
display(silver_df.count())
# display(silver_df.groupBy('event_type').count())

716

In [0]:
silver_df.filter(col('duration')<0).show()
display(silver_df.groupBy('event_type').count())
silver_df.select('score').distinct().show()
silver_df.filter(col('timestamp').isNull() | col('created_at').isNull()).count()

+---+-------+----------+--------+--------+----------+----------+---+-------+--------+---------+-----------+-----+----------+----------+-----------+--------+
| id|user_id|event_type|page_url|referrer|session_id|created_at| os|browser|language|timestamp|device_type|score|skill_name|skill_type|skill_level|duration|
+---+-------+----------+--------+--------+----------+----------+---+-------+--------+---------+-----------+-----+----------+----------+-----------+--------+
+---+-------+----------+--------+--------+----------+----------+---+-------+--------+---------+-----------+-----+----------+----------+-----------+--------+



event_type,count
page_view,571
quiz_completed,16
skill_added,21
session_end,95
logout,13


+-----+
|score|
+-----+
|100.0|
| 75.0|
| 25.0|
| NULL|
+-----+



0

In [0]:
display(silver_df.groupBy('id').count().filter(col('count')>1))
display(silver_df.filter(col('id').isNull()).count())
display(silver_df.select(['user_id','created_at']).distinct().orderBy('created_at',ascending=False).limit(15))

id,count


0

user_id,created_at
null,2026-10-05T08:53:30.000Z
null,2026-10-05T08:52:31.000Z
null,2026-10-04T09:14:44.000Z
null,2026-10-04T09:13:54.000Z
null,2026-10-04T09:13:53.000Z
null,2026-10-04T09:13:46.000Z
null,2026-10-04T09:07:47.000Z
null,2026-10-04T09:07:42.000Z
null,2026-10-04T09:07:36.000Z
1,2026-10-04T09:07:35.000Z


In [0]:
display(silver_df.groupBy('user_id').count())

user_id,count
null,170
1,445
2,9
3,3
4,4
5,43
6,5
7,26
8,5
9,6


In [0]:
silver_df.write.format('delta').mode('overwrite').saveAsTable(f'{catalog_name}.silver.slv_metadata')